## Configurações, Imports e Funções

### Imports

In [ ]:
import os
import re
import json
import math
import warnings
from glob import glob
from collections import defaultdict, Counter
import ast
import time
import joblib
import pickle

import pandas as pd
import numpy as np
from pandarallel import pandarallel

# Gensim
from gensim import corpora, models
from gensim.models import LdaModel
from gensim.corpora import Dictionary

# Scikit-learn
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score

# XGBoost
from xgboost import XGBClassifier

# Imbalanced-learn
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

### Configurações

In [ ]:
# Configurações
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings(
    "ignore",
    message="Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated",
    category=FutureWarning
)
pandarallel.initialize(progress_bar=True)

# Constantes de configuração
RANDOM_STATE = 42
NUMERO_DE_TOPICOS = 5
WORDS_PER_TOPIC_FOR_DISPLAY = 8

# Caminhos dos arquivos
DATASET_PATH = './datasets/database-lemmetizado.csv.zip'
FILTERED_IPCR_PATH = './datasets/database-filtrado.csv.zip'
MODEL_ML_DIR = f'./modelos/ml/'
MODELS_DIR = f'./modelos/lda_por_ano/{NUMERO_DE_TOPICOS}_topics/'
RESULTS_DIR = f'./resultados_por_ano/{NUMERO_DE_TOPICOS}_topics/'
ANALYSIS_DIR = f'./resultados_analise/{NUMERO_DE_TOPICOS}_topics/'


### Funções

#### Funções Auxiliares

In [ ]:
def load_dataset(filepath, compression='zip'):
    """Carrega o dataset e prepara para processamento."""
    df = pd.read_csv(filepath, compression=compression)
    df = df.sort_values('date_published').reset_index(drop=True)
    df['date_published'] = pd.to_datetime(df['date_published'], errors='coerce')
    return df

def sample_reduce_by_year(df, keep_fraction=0.25, seed=42, year_col='year', min_per_year=1):
    """
    Retorna uma versão reduzida do DataFrame mantendo 'keep_fraction' das linhas
    de cada ano (ou seja, remove ~ (1 - keep_fraction) por ano).
    - keep_fraction: proporção a manter por ano (0 < keep_fraction <= 1). Padrão 0.25 (remove 75%).
    - seed: semente para reprodutibilidade.
    - year_col: nome da coluna com o ano.
    - min_per_year: garante pelo menos este número de linhas por ano.
    """
    if year_col not in df.columns:
        raise ValueError(f"Coluna '{year_col}' não encontrada no DataFrame.")
    if not (0 < keep_fraction <= 1):
        raise ValueError("keep_fraction deve ser > 0 e <= 1.")

    sampled_parts = []
    grouped = df.groupby(year_col)
    for _, g in grouped:
        k = max(min_per_year, int(round(len(g) * keep_fraction)))
        k = min(k, len(g))
        if k > 0:
            sampled_parts.append(g.sample(n=k, random_state=seed))

    if not sampled_parts:
        return pd.DataFrame(columns=df.columns)

    sampled = pd.concat(sampled_parts, ignore_index=False)
    sampled = sampled.sort_index()
    return sampled.reset_index(drop=True)

def tokenize_text(text):
    """Tokeniza o texto convertendo para minúsculas e separando por espaços."""
    return text.lower().split()


def prepare_corpus_and_dictionary(df, token_column='tokens'):
    """Prepara corpus e dicionário para modelagem LDA."""
    documents = df[token_column].dropna().astype(str).tolist()
    processed_docs = [tokenize_text(doc) for doc in documents]
    
    dictionary = corpora.Dictionary(processed_docs)
    corpus = [dictionary.doc2bow(doc) for doc in processed_docs]
    
    return corpus, dictionary, processed_docs

#### Funções de Treinamento do LDA

In [ ]:
def train_and_save_lda_per_year(df, num_topics=10, passes=10, 
                                alpha='auto', eta='auto', output_dir=MODELS_DIR):
    """
    Treina um modelo LDA para cada ano e salva-o individualmente.
    
    Args:
        df: DataFrame com 'date_published' e 'tokens'
        num_topics: Número de tópicos para cada modelo
        passes: Número de passes de treinamento
        alpha: Parâmetro alpha do LDA
        eta: Parâmetro eta do LDA
        output_dir: Diretório para salvar os modelos
    """
    df['year'] = df['date_published'].dt.year
    years = sorted(df['year'].dropna().unique())
    
    print(f"Anos encontrados no dataset: {years}")
    
    os.makedirs(output_dir, exist_ok=True)
    print(f"Modelos serão salvos em: '{output_dir}'")
    
    models_trained_count = 0
    
    for year in years:
        print(f"\n=== Processando ano: {year} ===")
        
        df_year = df[df['year'] == year].copy()
        documents = df_year['tokens'].dropna().tolist()
        
        processed_docs = [
            doc.split() if isinstance(doc, str) else doc 
            for doc in documents
        ]
        processed_docs = [doc for doc in processed_docs if doc]
        
        print(f"Documentos para treinamento: {len(processed_docs)}")
        
        dictionary_year = corpora.Dictionary(processed_docs)
        corpus_year = [dictionary_year.doc2bow(doc) for doc in processed_docs]
        
        try:
            lda_model = models.LdaModel(
                corpus=corpus_year,
                id2word=dictionary_year,
                num_topics=num_topics,
                random_state=RANDOM_STATE,
                passes=passes,
                alpha=alpha,
                eta=eta
            )
            
            model_path = os.path.join(output_dir, f'lda_model_{year}.model')
            lda_model.save(model_path)
            
            print(f"Modelo salvo: '{model_path}'")
            models_trained_count += 1
            
        except Exception as e:
            print(f"ERRO ao treinar o modelo para o ano {year}: {e}")
            continue
    
    print(f"\n=== {models_trained_count} modelos treinados com sucesso ===")


def display_topics_from_models_with_probs(model_dir, num_words=10):
    """Exibe os tópicos de cada modelo LDA com probabilidades."""
    if not os.path.exists(model_dir):
        print(f"ERRO: O diretório '{model_dir}' não foi encontrado.")
        return

    model_files = [
        f for f in os.listdir(model_dir) 
        if f.startswith('lda_model_') and f.endswith('.model')
    ]
    
    if not model_files:
        print(f"Nenhum modelo encontrado em '{model_dir}'.")
        return
    
    model_files.sort()
    print(f"Encontrados {len(model_files)} modelos.\n")
    
    for filename in model_files:
        try:
            match = re.search(r'_(\d{4})\.model', filename)
            if not match:
                continue
            
            year = match.group(1)
            model_path = os.path.join(model_dir, filename)
            lda_model = LdaModel.load(model_path)
            
            print(f"--- Tópicos para o Ano: {year} ---")
            
            topics = lda_model.show_topics(
                num_topics=-1, 
                num_words=num_words, 
                formatted=False
            )
            
            for topic_id, word_probs in topics:
                formatted_words = [
                    f"{word} ({prob*100:.2f}%)" 
                    for word, prob in word_probs
                ]
                print(f"Tópico {topic_id}: {', '.join(formatted_words)}")
            print()
                
        except Exception as e:
            print(f"ERRO ao processar {filename}: {e}")

#### Funções de processamento de patentes

In [ ]:
def _processar_patente(row, lda_model, dicionario, 
                      distribuicoes_topicos_ano, num_topics):
    """Processa uma única patente extraindo distribuição de tópicos."""
    lens_id_final = row['lens_id']
    tokens_brutos = row['tokens']

    # filtros de tokens inválidos
    if pd.isna(tokens_brutos) or tokens_brutos == '':
        return None
    
    if isinstance(tokens_brutos, str):
        tokens_patente = tokens_brutos.split()
    elif isinstance(tokens_brutos, list):
        tokens_patente = tokens_brutos
    else:
        return None
    
    tokens_patente = [t for t in tokens_patente if t and isinstance(t, str)]
    
    if not tokens_patente:
        return None

    # distribuição de tópicos para a patente
    doc_bow = dicionario.doc2bow(tokens_patente)
    distribuicao_topicos_patente = lda_model.get_document_topics(
        doc_bow, minimum_probability=0.0
    )
    prob_topico_dado_doc = {
        topico_id: prob 
        for topico_id, prob in distribuicao_topicos_patente
    }

    resultado_patente = {
        'lens_id': lens_id_final, 
        'year': int(row['year'])
    }
    
    # calcula a pontuação das palavras por tópico
    for id_topico in range(num_topics):
        p_topico_na_patente = prob_topico_dado_doc.get(id_topico, 0)
        distribuicao_palavras_topico = distribuicoes_topicos_ano[id_topico]
        
        lista_pontuacao_palavras = []
        doc_bow_dict = dict(doc_bow)
        
        for id_palavra, p_palavra_no_topico in distribuicao_palavras_topico:
            if id_palavra in doc_bow_dict:
                pontuacao = float(p_topico_na_patente * p_palavra_no_topico)
                palavra_str = dicionario[id_palavra]
                lista_pontuacao_palavras.append((palavra_str, pontuacao))
        
        lista_pontuacao_palavras.sort(key=lambda item: item[1], reverse=True)
        top_palavras = lista_pontuacao_palavras[:50]
        
        resultado_patente[f'Topic_{id_topico}'] = json.dumps(
            top_palavras, ensure_ascii=False
        )
    
    return resultado_patente


def analisar_e_salvar_por_ano_aprimorado(df, modelos_dir, num_topics, 
                                         output_dir, anos_para_rodar=None, 
                                         ignorar_existentes=True):
    """Processa patentes ano por ano de forma otimizada."""
    os.makedirs(output_dir, exist_ok=True)
    
    colunas_necessarias = ['lens_id', 'year', 'tokens']
    for col in colunas_necessarias:
        if col not in df.columns:
            raise ValueError(f"Coluna '{col}' não encontrada!")
    
    if df['lens_id'].duplicated().any():
        duplicados = df['lens_id'].duplicated().sum()
        print(f"AVISO: {duplicados} lens_id duplicados. Removendo...")
        df = df.drop_duplicates(subset=['lens_id'], keep='first')
    
    if anos_para_rodar:
        anos_alvo = (
            [anos_para_rodar] if isinstance(anos_para_rodar, int) 
            else sorted(anos_para_rodar)
        )
    else:
        anos_alvo = sorted(df['year'].dropna().unique().astype(int))
    
    print(f"Iniciando análise para os anos: {anos_alvo}")

    for ano in anos_alvo:
        if ignorar_existentes:
            caminho_parquet = os.path.join(output_dir, f'resultados_{ano}.parquet')
            caminho_csv = os.path.join(output_dir, f'resultados_{ano}.csv')
            
            if os.path.exists(caminho_parquet) or os.path.exists(caminho_csv):
                print(f"\n--- Ano {ano} já processado. Pulando... ---")
                continue

        print(f"\n{'='*60}")
        print(f"Processando o ano: {ano}")
        print(f"{'='*60}")
        
        caminho_modelo = os.path.join(modelos_dir, f'lda_model_{ano}.model')
        
        if not os.path.exists(caminho_modelo):
            print(f"AVISO: Modelo para {ano} não encontrado")
            continue
            
        try:
            lda_model_ano = LdaModel.load(caminho_modelo)
            dicionario_ano = lda_model_ano.id2word
            print("Modelo carregado")
        except Exception as e:
            print(f"ERRO ao carregar modelo: {e}")
            continue

        print("Pré-calculando distribuições de tópicos...")
        distribuicoes_topicos_ano = {
            id_topico: lda_model_ano.get_topic_terms(id_topico, topn=500)
            for id_topico in range(num_topics)
        }
        
        df_ano = df[df['year'] == ano].copy()
        print(f"{len(df_ano)} patentes para processar")

        if df_ano.empty:
            print("Nenhuma patente para processar.")
            continue
        
        print("Processando patentes...")
        try:
            resultados_do_ano_series = df_ano.parallel_apply(
                _processar_patente, 
                axis=1, 
                lda_model=lda_model_ano, 
                dicionario=dicionario_ano,
                distribuicoes_topicos_ano=distribuicoes_topicos_ano, 
                num_topics=num_topics
            )
        except Exception as e:
            print(f"ERRO: {e}")
            import traceback
            traceback.print_exc()
            continue
        
        resultados_do_ano_lista = [
            res for res in resultados_do_ano_series if res is not None
        ]
        
        if not resultados_do_ano_lista:
            print("Nenhum resultado válido.")
            continue
        
        print(f"{len(resultados_do_ano_lista)} patentes processadas")
            
        df_resultados_ano = pd.DataFrame(resultados_do_ano_lista)
        
        caminho_saida_parquet = os.path.join(output_dir, f'resultados_{ano}.parquet')
        caminho_saida_csv = os.path.join(output_dir, f'resultados_{ano}.csv')
        
        try:
            df_resultados_ano.to_parquet(
                caminho_saida_parquet, index=False, engine='pyarrow'
            )
            print(f"✓ Salvo em Parquet: {caminho_saida_parquet}")
        except Exception as e:
            print(f"⚠ Erro ao salvar Parquet, usando CSV: {str(e)[:100]}")
            try:
                df_resultados_ano.to_csv(caminho_saida_csv, index=False)
                print(f"✓ Salvo em CSV: {caminho_saida_csv}")
            except Exception as e2:
                print(f"✗ ERRO ao salvar: {e2}")

    print("\n" + "="*60)
    print("PROCESSAMENTO CONCLUÍDO")
    print("="*60)


#### Funções de Tratamento de Resultados

In [ ]:
def concatenar_resultados_por_ano(diretorio_resultados, formato='csv'):
    """Concatena todos os arquivos de resultados em um único DataFrame."""
    if not os.path.exists(diretorio_resultados):
        raise ValueError(f"Diretório '{diretorio_resultados}' não encontrado!")
    
    padroes_formato = {
        'auto': ['resultados_*.csv', 'resultados_*.parquet'],
        'csv': ['resultados_*.csv'],
        'parquet': ['resultados_*.parquet']
    }
    
    padroes = padroes_formato.get(formato)
    if not padroes:
        raise ValueError("formato deve ser 'csv', 'parquet' ou 'auto'")
    
    arquivos = []
    for padrao in padroes:
        caminho_completo = os.path.join(diretorio_resultados, padrao)
        arquivos.extend(glob(caminho_completo))
    
    if not arquivos:
        print(f"⚠ Nenhum arquivo encontrado em '{diretorio_resultados}'")
        return pd.DataFrame()
    
    arquivos = sorted(set(arquivos))
    print(f"Encontrados {len(arquivos)} arquivos para concatenar")
    print(f"{'='*60}")
    
    dataframes_lista = []
    
    for arquivo in arquivos:
        try:
            nome_arquivo = os.path.basename(arquivo)
            
            if arquivo.endswith('.csv'):
                df_temp = pd.read_csv(arquivo)
                tipo = 'CSV'
            elif arquivo.endswith('.parquet'):
                df_temp = pd.read_parquet(arquivo)
                tipo = 'Parquet'
            else:
                continue
            
            dataframes_lista.append(df_temp)
            print(f"✓ {nome_arquivo} ({tipo}) - {len(df_temp)} registros")
            
        except Exception as e:
            print(f"✗ ERRO ao carregar {nome_arquivo}: {str(e)[:80]}")
            continue
    
    if not dataframes_lista:
        print("\n⚠ Nenhum DataFrame válido foi carregado!")
        return pd.DataFrame()
    
    print(f"\n{'='*60}")
    print(f"Concatenando {len(dataframes_lista)} DataFrames...")
    
    df_final = pd.concat(dataframes_lista, ignore_index=True)
    
    print("Concatenação concluída!")
    print(f"\nRESUMO DO DATASET FINAL:")
    print(f"{'='*60}")
    print(f" --- Total de registros: {len(df_final):,}")
    print(f" --- Total de colunas: {len(df_final.columns)}")
    print(f" --- Anos presentes: {sorted(df_final['year'].unique().tolist())}")
    print(f" --- Registros por ano:")
    
    contagem_por_ano = df_final['year'].value_counts().sort_index()
    for ano, contagem in contagem_por_ano.items():
        print(f"    - {ano}: {contagem:,} patentes")
    
    print(f"{'='*60}")
    
    return df_final


def converter_json_para_listas(df, prefixo_coluna='Topic_'):
    """Converte colunas JSON (strings) de volta para listas Python."""
    df_copia = df.copy()
    colunas_topicos = [
        col for col in df_copia.columns if col.startswith(prefixo_coluna)
    ]
    
    print(f"Convertendo {len(colunas_topicos)} colunas de tópicos...")
    
    for col in colunas_topicos:
        try:
            df_copia[col] = df_copia[col].apply(
                lambda x: json.loads(x) 
                if isinstance(x, str) and x.strip() != '[]' 
                else []
            )
        except Exception as e:
            print(f"⚠ Erro ao converter coluna {col}: {e}")
            continue
    
    print("Conversão concluída!")
    return df_copia

def _compute_dominant_topic(row, topic_cols):
    best_topic = None
    best_score = -1.0
    for col in topic_cols:
        lst = row.get(col, [])
        # lst esperado como lista de [palavra, score] ou (palavra, score)
        total = 0.0
        if isinstance(lst, str):
            try:
                lst = json.loads(lst)
            except Exception:
                lst = []
        if isinstance(lst, list):
            for item in lst:
                try:
                    score = float(item[1])
                except Exception:
                    score = 0.0
                total += score
        if total > best_score:
            best_score = total
            best_topic = col
    return best_topic

#### Funções de Análise de Similaridade

In [ ]:
def clean_word_original(palavra_suja):
    """Limpa palavras removendo caracteres não-alfabéticos."""
    if isinstance(palavra_suja, str):
        palavra_limpa = re.sub(r"[^a-zA-Zá-úÁ-Ú]", "", palavra_suja)
        return palavra_limpa if palavra_limpa else None
    return None

# Analisa de forma a fazer a pergunta:
# Estas patentes têm as mesmas palavras (sem considerar seus pesos)?
def analise_similaridade_jaccard(df_completo, numero_palavras_por_topico, 
                                 percentual_similaridade, anos_a_frente=1):
    """Análise de similaridade usando coeficiente de Jaccard otimizado."""
    df_otimizado = df_completo.copy()
    topic_columns = [col for col in df_otimizado.columns if 'Topic_' in col]
    
    print("Iniciando extração vetorizada de palavras-chave...")

    # 'Melt' transforma colunas (Topic_1, Topic_2) em linhas
    df_melted = df_otimizado.melt(
        id_vars=['lens_id'], 
        value_vars=topic_columns, 
        value_name='topic_list'
    )
    
    df_melted = df_melted.dropna(subset=['topic_list'])

    # 'Explode' transforma listas em linhas
    # Se 'topic_list' era [['word1', 0.5], ['word2', 0.4]],
    # agora teremos duas linhas: ['word1', 0.5] e ['word2', 0.4]
    df_exploded = df_melted.explode('topic_list')
    
    df_exploded = df_exploded[
        df_exploded['topic_list'].apply(
            lambda x: isinstance(x, list) and len(x) > 0
        )
    ]
    
    df_exploded['palavra_suja'] = df_exploded['topic_list'].str[0]
    
    print("Limpando palavras-chave...")
    df_exploded['palavra_limpa'] = df_exploded['palavra_suja'].apply(
        clean_word_original
    )
    
    df_exploded = df_exploded.dropna(subset=['palavra_limpa'])

    # Agrupa por patente E por tópico de origem, e pega as N primeiras
    # Isso simula o '[:n_palavras]' da função original, para CADA tópico
    df_top_n = df_exploded.groupby(['lens_id', 'variable']).head(
        numero_palavras_por_topico
    )
    
    palavras_chave_series = df_top_n.groupby('lens_id')['palavra_limpa'].apply(set)
    df_otimizado['palavras_chave'] = df_otimizado['lens_id'].map(
        palavras_chave_series
    )
    df_otimizado['palavras_chave'] = df_otimizado['palavras_chave'].apply(
        lambda x: x if isinstance(x, set) else set()
    )
    
    print("Assinaturas de palavras criadas com sucesso.")
    
    print("\nIniciando comparação de similaridade otimizada...")
    
    resultados_finais = []
    anos_unicos = sorted(df_otimizado['year'].unique())
    
    for i in range(len(anos_unicos)):
        ano_atual = anos_unicos[i]
        
        # Verifica se há anos suficientes à frente
        if i + anos_a_frente >= len(anos_unicos):
            # Adiciona patentes do último ano(s) sem comparação
            df_ano_atual = df_otimizado[df_otimizado['year'] == ano_atual]
            for patente_atual in df_ano_atual.itertuples():
                resultados_finais.append({
                    'lens_id': patente_atual.lens_id,
                    'patentes_similares': []
                })
            continue
        
        # Coleta anos futuros para comparação
        anos_futuros = anos_unicos[i+1:i+1+anos_a_frente]
        print(f"Comparando ano {ano_atual} com anos {anos_futuros}...")
        
        df_ano_atual = df_otimizado[df_otimizado['year'] == ano_atual]
        df_anos_futuros = df_otimizado[df_otimizado['year'].isin(anos_futuros)]
        
        # Cria um mapa de lookup rápido para os anos futuros
        palavras_futuras_map = df_anos_futuros.set_index('lens_id')['palavras_chave']
        
        # Cria o Índice Invertido para os anos futuros
        inverted_index = defaultdict(set)
        for lens_id, palavras_set in palavras_futuras_map.items():
            for palavra in palavras_set:
                inverted_index[palavra].add(lens_id)

        # Itera sobre o ano atual        
        for patente_atual in df_ano_atual.itertuples():
            id_atual = patente_atual.lens_id
            palavras_atuais = patente_atual.palavras_chave
            len_atuais = len(palavras_atuais)
            
            if len_atuais == 0:
                resultados_finais.append({
                    'lens_id': id_atual,
                    'patentes_similares': []
                })
                continue
            
            # Encontra candidatos usando o Índice Invertido
            # 'candidate_counts' irá armazenar: {id_candidato -> contagem_de_palavras_em_comum}
            # A contagem de palavras em comum é exatamente o tamanho da INTERSEÇÃO
            candidate_counts = Counter()
            for palavra in palavras_atuais:
                candidate_counts.update(inverted_index[palavra])
                
            patentes_similares_encontradas = []

            for id_futuro, intersecao in candidate_counts.items():
                palavras_futuras = palavras_futuras_map[id_futuro]
                uniao = len_atuais + len(palavras_futuras) - intersecao
                
                similaridade = intersecao / uniao if uniao > 0 else 0
                
                if similaridade >= percentual_similaridade:
                    patentes_similares_encontradas.append(id_futuro)

            resultados_finais.append({
                'lens_id': id_atual,
                'patentes_similares': patentes_similares_encontradas
            })
    
    print("Comparação finalizada.")
    
    df_resultados = pd.DataFrame(resultados_finais)
    df_otimizado = pd.merge(df_otimizado, df_resultados, on='lens_id', how='left')
    
    df_otimizado['patentes_similares'] = df_otimizado['patentes_similares'].apply(
        lambda x: x if isinstance(x, list) else []
    )
    df_otimizado['count_similares'] = df_otimizado['patentes_similares'].apply(len)
    df_otimizado['emergente'] = df_otimizado['count_similares'] > 0
    
    print("\nResultado Final:")
    print(df_otimizado[['lens_id', 'year', 'patentes_similares', 
                        'count_similares', 'emergente']].head())
    
    os.makedirs(f'{ANALYSIS_DIR}', exist_ok=True)
    output_path = (
        f'{ANALYSIS_DIR}analise-de-similaridade_jaccard_'
        f'n{numero_palavras_por_topico}_s{percentual_similaridade:.2f}_'
        f'anos{anos_a_frente}.csv'
    )
    df_otimizado.to_csv(output_path, index=True)
    
    print(f"\nArquivo salvo: {output_path}")
    
    return df_otimizado

    """Calcula similaridade de Tanimoto entre dois vetores de características."""
    palavras_comuns = set(vetor1.keys()).intersection(set(vetor2.keys()))
    produto_escalar = sum(vetor1[palavra] * vetor2[palavra] for palavra in palavras_comuns)
    
    if produto_escalar == 0.0:
        return 0.0
    
    mag_quad_vetor1 = sum(prob**2 for prob in vetor1.values())
    mag_quad_vetor2 = sum(prob**2 for prob in vetor2.values())
    
    denominador = mag_quad_vetor1 + mag_quad_vetor2 - produto_escalar
    
    if denominador == 0:
        return 0.0
    
    return produto_escalar / denominador

# Cosseno: encontrar patentes que falam sobre as mesmas coisas, 
# sem se importar com a força da presença das palavras, e sim dos temas.
# Tanimoto: encontrar patentes que falam sobre as mesmas coisas,
# e que são intrinsecamente muito parecidas em seus focos.
def analise_similaridade_tanimoto_or_cosseno(df_completo, numero_palavras_por_topico, 
                                  percentual_similaridade, metodo='tanimoto',
                                  anos_a_frente=1):
    """Análise de similaridade usando Tanimoto/Cossenos otimizado."""
    df2 = df_completo.copy()
    topic_columns = [col for col in df2.columns if 'Topic_' in col]
    
    print("Iniciando extração vetorizada de vetores de características...")
    
    # 'Melt' transforma colunas (Topic_1, Topic_2) em linhas
    df_melted = df2.melt(
        id_vars=['lens_id'], 
        value_vars=topic_columns, 
        var_name='topic_source',
        value_name='topic_list'
    )
    
    df_melted = df_melted.dropna(subset=['topic_list'])

    # 'Explode' transforma listas em linhas
    df_exploded = df_melted.explode('topic_list')
    df_exploded = df_exploded.dropna(subset=['topic_list'])
    
    df_exploded['is_valid'] = df_exploded['topic_list'].apply(
        lambda x: isinstance(x, list) and len(x) == 2
    )
    df_exploded = df_exploded[df_exploded['is_valid']]
    
    df_exploded['palavra_suja'] = df_exploded['topic_list'].str[0].astype(str)
    df_exploded['probabilidade'] = pd.to_numeric(
        df_exploded['topic_list'].str[1], errors='coerce'
    )
    
    df_exploded = df_exploded.dropna(subset=['probabilidade'])
    
    df_exploded['palavra_limpa'] = df_exploded['palavra_suja'].str.replace(
        r"[^a-zA-Zá-úÁ-Ú]", "", regex=True
    )
    
    df_exploded = df_exploded[df_exploded['palavra_limpa'] != '']
    
    df_top_n = df_exploded.groupby(['lens_id', 'topic_source']).head(
        numero_palavras_por_topico
    )
    
    df_final_palavras = df_top_n.drop_duplicates(
        subset=['lens_id', 'palavra_limpa'], keep='first'
    )
    
    vetores_series = df_final_palavras.groupby('lens_id').apply(
        lambda x: dict(zip(x['palavra_limpa'], x['probabilidade']))
    )
    
    df2['vetor_caracteristicas'] = df2['lens_id'].map(vetores_series)
    df2['vetor_caracteristicas'] = df2['vetor_caracteristicas'].apply(
        lambda x: x if isinstance(x, dict) else {}
    )
    
    print("Vetores criados com sucesso.")
    
    del df_melted, df_exploded, df_top_n, df_final_palavras, vetores_series
    
    print(f"Pré-calculando magnitudes para o método '{metodo}'...")
    
    if metodo == 'cossenos':
        df2['magnitude'] = df2['vetor_caracteristicas'].apply(
            lambda v: math.sqrt(sum(prob**2 for prob in v.values()))
        )
    elif metodo == 'tanimoto':
        df2['mag_quadrada'] = df2['vetor_caracteristicas'].apply(
            lambda v: sum(prob**2 for prob in v.values())
        )
    else:
        raise ValueError("Método desconhecido para similaridade.")
    
    print("Magnitudes calculadas.")
    
    print(f"\nIniciando comparação de similaridade ({metodo}) otimizada...")
    
    resultados_finais = []
    anos_unicos = sorted(df2['year'].unique())
    
    for i in range(len(anos_unicos)):
        ano_atual = anos_unicos[i]
        
        # Verifica se há anos suficientes à frente
        if i + anos_a_frente >= len(anos_unicos):
            df_ano_atual = df2[df2['year'] == ano_atual]
            for patente_atual in df_ano_atual.itertuples():
                resultados_finais.append({
                    'lens_id': patente_atual.lens_id,
                    'patentes_similares': []
                })
            continue
        
        # Coleta anos futuros para comparação
        anos_futuros = anos_unicos[i+1:i+1+anos_a_frente]
        print(f"Comparando ano {ano_atual} com anos {anos_futuros}...")
        
        df_ano_atual = df2[df2['year'] == ano_atual]
        df_anos_futuros = df2[df2['year'].isin(anos_futuros)]
        
        # Cria mapas de lookup rápidos para o ano seguinte
        vetores_futuros_map = df_anos_futuros.set_index('lens_id')['vetor_caracteristicas']
        mag_futuros_map = df_anos_futuros.set_index('lens_id')[
            'magnitude' if metodo == 'cossenos' else 'mag_quadrada'
        ]
        
        # Cria o Índice Invertido Ponderado para o ano seguinte
        # Formato: {palavra -> {id1: prob1, id2: prob2, ...}}
        inverted_index = defaultdict(dict)
        for lens_id, vetor in vetores_futuros_map.items():
            for palavra, prob in vetor.items():
                inverted_index[palavra][lens_id] = prob

        # Itera sobre o ano atual        
        for patente_atual in df_ano_atual.itertuples():
            id_atual = patente_atual.lens_id
            vetor_atual = patente_atual.vetor_caracteristicas
            
            if not vetor_atual:
                resultados_finais.append({
                    'lens_id': id_atual, 
                    'patentes_similares': []
                })
                continue

            # Calcula os produtos escalares para TODOS os candidatos de uma vez
            # 'dot_products' armazenará: {id_candidato -> produto_escalar}
            dot_products = Counter()
            for palavra, prob_atual in vetor_atual.items():
                if palavra in inverted_index:
                    for id_futuro, prob_futuro in inverted_index[palavra].items():
                        dot_products[id_futuro] += prob_atual * prob_futuro
                
            patentes_similares_encontradas = []
            
            if metodo == 'cossenos':
                mag_atual = patente_atual.magnitude
                if mag_atual == 0:
                    continue
                
                for id_futuro, produto_escalar in dot_products.items():
                    mag_futuro = mag_futuros_map[id_futuro]
                    if mag_futuro == 0:
                        continue
                    
                    similaridade = produto_escalar / (mag_atual * mag_futuro)
                    if similaridade >= percentual_similaridade:
                        patentes_similares_encontradas.append(id_futuro)
            
            elif metodo == 'tanimoto':
                mag_sq_atual = patente_atual.mag_quadrada
                
                for id_futuro, produto_escalar in dot_products.items():
                    mag_sq_futuro = mag_futuros_map[id_futuro]
                    denominador = mag_sq_atual + mag_sq_futuro - produto_escalar
                    
                    if denominador == 0:
                        continue
                    
                    similaridade = produto_escalar / denominador
                    if similaridade >= percentual_similaridade:
                        patentes_similares_encontradas.append(id_futuro)
            
            resultados_finais.append({
                'lens_id': id_atual,
                'patentes_similares': patentes_similares_encontradas
            })
    
    print("Comparação finalizada.")
    
    df_resultados = pd.DataFrame(resultados_finais)
    df2 = pd.merge(df2, df_resultados, on='lens_id', how='left')
    
    df2['patentes_similares'] = df2['patentes_similares'].apply(
        lambda x: x if isinstance(x, list) else []
    )
    df2['count_similares'] = df2['patentes_similares'].apply(len)
    df2['emergente'] = df2['count_similares'] > 0
    
    if 'magnitude' in df2.columns:
        df2 = df2.drop(columns=['magnitude'])
    if 'mag_quadrada' in df2.columns:
        df2 = df2.drop(columns=['mag_quadrada'])
    
    print("\nResultado Final:")
    print(df2[['lens_id', 'year', 'patentes_similares', 
               'count_similares', 'emergente']].head())
    
    output_path = (
        f'{ANALYSIS_DIR}analise-de-similaridade_{metodo}_'
        f'n{numero_palavras_por_topico}_s{percentual_similaridade:.2f}'
        f'_anos{anos_a_frente}.csv'
    )
    df2.to_csv(output_path, index=True)
    
    print(f"\n✓ Arquivo salvo: {output_path}")
    
    return df2

def analise_emergencia_patente_topico(df_completo, numero_palavras_por_topico, 
                                      percentual_similaridade, metodo='tanimoto', 
                                      anos_a_frente=1):
    """
    Analisa a emergência de PATENTES (Micro) para identificar patentes-semente.
    
    Compara uma patente de um ano T (baseada no seu 'dominant_topic') com
    todos os TÓPICOS de anos futuros (T+1, T+2...).
    
    Se a patente for altamente similar (>= percentual_similaridade) a QUALQUER
    tópico futuro, ela é marcada como 'emergente'.
    
    Argumentos:
    df_completo -- O DataFrame original (deve ter 'lens_id', 'year', 
                                         'dominant_topic' e as colunas 'Topic_X')
    numero_palavras_por_topico -- N palavras para definir a assinatura do tópico.
    metodo -- 'tanimoto' ou 'cossenos'.
    anos_a_frente -- Janela futura para procurar similaridade (ex: 1, 2, 3 anos).
    percentual_similaridade -- Limiar de ALTA similaridade para considerar uma
                         patente como emergente (ex: > 0.5).
                        
    Retorna:
    Um DataFrame com os resultados por patente, com a coluna 'emergente'.
    """
    def calcular_similaridade_cosseno(vet1, mag1, vet2, mag2):
        """Calcula a similaridade de cossenos entre dois vetores (dicionários)."""
        # Produto escalar
        dot_product = 0
        for palavra, prob1 in vet1.items():
            if palavra in vet2:
                dot_product += prob1 * vet2[palavra]
                
        if mag1 == 0 or mag2 == 0:
            return 0
            
        return dot_product / (mag1 * mag2)

    def calcular_similaridade_tanimoto(vet1, mag1, vet2, mag2):
        """Calcula a similaridade Tanimoto (Jaccard) entre dois vetores (dicionários)."""
        # mag1 e mag2 já estão pré-calculados como (sum(p**2))
        
        # Produto escalar
        dot_product = 0
        for palavra, prob1 in vet1.items():
            if palavra in vet2:
                dot_product += prob1 * vet2[palavra]

        denominador = mag1 + mag2 - dot_product
        
        if denominador == 0:
            return 0
            
        return dot_product / denominador

    df_macro = df_completo.copy()
    topic_columns = [col for col in df_macro.columns if 'Topic_' in col]
    
    # --- ETAPA 1: CRIAÇÃO DOS VETORES DE TÓPICOS (Inalterada) ---
    # Esta parte é idêntica à função original. Ela cria o "mapa" de vetores
    # para todos os tópicos (Topic_1, Topic_2...) de todos os anos.
    
    print("Iniciando extração de vetores de TÓPICOS (Macro)...")
    
    # Usamos drop_duplicates para obter a definição de cada Tópico por Ano
    # (Assumindo que a definição do Tópico é a mesma para todas as patentes
    # daquele ano, como na função original)
    df_topicos_ano = df_macro[['year'] + topic_columns].drop_duplicates(
        subset=['year'], keep='first'
    )
    
    df_melted = df_topicos_ano.melt(
        id_vars=['year'], 
        value_vars=topic_columns, 
        var_name='topic_source',  # Ex: 'Topic_1', 'Topic_2'
        value_name='topic_list'
    )
    
    df_melted = df_melted.dropna(subset=['topic_list'])
    
    # Explode e limpa as palavras-chave
    df_exploded = df_melted.explode('topic_list')
    df_exploded = df_exploded.dropna(subset=['topic_list'])
    
    df_exploded['is_valid'] = df_exploded['topic_list'].apply(
        lambda x: isinstance(x, list) and len(x) == 2
    )
    df_exploded = df_exploded[df_exploded['is_valid']]
    
    df_exploded['palavra_suja'] = df_exploded['topic_list'].str[0].astype(str)
    df_exploded['probabilidade'] = pd.to_numeric(
        df_exploded['topic_list'].str[1], errors='coerce'
    )
    df_exploded = df_exploded.dropna(subset=['probabilidade'])
    
    df_exploded['palavra_limpa'] = df_exploded['palavra_suja'].str.replace(
        r"[^a-zA-Zá-úÁ-Ú]", "", regex=True
    )
    df_exploded = df_exploded[df_exploded['palavra_limpa'] != '']
    
    # Pega as N palavras mais importantes
    df_top_n = df_exploded.groupby(['year', 'topic_source']).head(
        numero_palavras_por_topico
    )
    
    print("Criando vetores-assinatura para cada Tópico/Ano...")
    
    vetores_series = df_top_n.groupby(['year', 'topic_source']).apply(
        lambda x: dict(zip(x['palavra_limpa'], x['probabilidade']))
    )
    
    # Dicionário principal de vetores: { (ano, 'Topic_X'): {'palavra': prob...} }
    vetores_de_topicos = vetores_series.to_dict()
    
    # Pré-calcula magnitudes
    magnitudes = {}
    if metodo == 'cossenos':
        for key, vet in vetores_de_topicos.items():
            magnitudes[key] = math.sqrt(sum(p**2 for p in vet.values()))
    elif metodo == 'tanimoto':
        for key, vet in vetores_de_topicos.items():
            magnitudes[key] = sum(p**2 for p in vet.values())
    
    print("Vetores de tópicos criados com sucesso.")

    if 'dominant_topic' not in df_macro.columns:
        print("\n--- ERRO ---")
        print("Não foi possível rotular as patentes.")
        print("Crie uma coluna 'dominant_topic' no seu DataFrame 'df_completo'.")
        print("-----------------")
        return None
    
    print("\nIniciando análise de Patente-Tópico (Patente T -> Tópico T+)...")
    
    patentes_emergentes_ids = set() # Armazena os 'lens_id' emergentes
    anos_unicos = sorted(df_macro['year'].unique())
    
    for i in range(len(anos_unicos)):
        ano_atual = anos_unicos[i]
        
        if i + anos_a_frente >= len(anos_unicos):
            continue
            
        anos_futuros = anos_unicos[i+1 : i+1+anos_a_frente]
        
        # Pega TÓPICOS futuros
        keys_futuros = [k for k in vetores_de_topicos if k[0] in anos_futuros]
        
        # Pega PATENTES atuais (apenas as que têm um tópico dominante)
        patentes_atuais = df_macro[
            (df_macro['year'] == ano_atual) & 
            (df_macro['dominant_topic'].notna())
        ]
        
        if not keys_futuros or patentes_atuais.empty:
            continue
            
        print(f"Comparando {len(patentes_atuais)} patentes de {ano_atual} com os tópicos de {anos_futuros}...")

        # Pré-filtra os vetores futuros para este loop (otimização)
        vetores_futuros_loop = {
            key: (vetores_de_topicos[key], magnitudes[key]) 
            for key in keys_futuros
        }

        # Itera por cada patente do ano atual
        for index, patente in patentes_atuais.iterrows():
            patente_id = patente['lens_id']
            
            # O "vetor" da patente é o vetor do seu tópico dominante
            key_patente = (patente['year'], patente['dominant_topic'])
            
            if key_patente not in vetores_de_topicos:
                continue # Patente não tem vetor (tópico inválido ou ausente)
                
            vet_patente = vetores_de_topicos[key_patente]
            mag_patente = magnitudes[key_patente]
            
            encontrou_similar_alta = False
            
            # Compara esta patente com TODOS os tópicos futuros
            for key_futuro, (vet_futuro, mag_futuro) in vetores_futuros_loop.items():
                
                if metodo == 'cossenos':
                    sim = calcular_similaridade_cosseno(vet_patente, mag_patente, 
                                                        vet_futuro, mag_futuro)
                else: # tanimoto
                    sim = calcular_similaridade_tanimoto(vet_patente, mag_patente, 
                                                         vet_futuro, mag_futuro)
                
                # Se a similaridade for ALTA, encontramos uma ligação.
                if sim >= percentual_similaridade:
                    encontrou_similar_alta = True
                    break # Otimização: Se encontrou um, a patente é emergente.
            
            if encontrou_similar_alta:
                patentes_emergentes_ids.add(patente_id)

    print("\nAnálise concluída.")
    
    print(f"Total de {len(patentes_emergentes_ids)} patentes únicas rotuladas como emergentes.")
    
    # Rotula o DataFrame final com base nos IDs encontrados
    df_macro['emergente'] = df_macro['lens_id'].apply(
        lambda x: x in patentes_emergentes_ids
    )

    print("\nResultado Final (Patentes rotuladas):")
    print(df_macro[df_macro['emergente'] == True][[
        'lens_id', 'year', 'dominant_topic', 'emergente'
    ]].head())

    os.makedirs(ANALYSIS_DIR, exist_ok=True)
    
    output_path = (
        f'{ANALYSIS_DIR}/analise-de-similaridade_topicos_{metodo}_'
        f'n{numero_palavras_por_topico}_s{percentual_similaridade:.2f}'
        f'_anos{anos_a_frente}.csv'
    )
    
    # Salva o DataFrame completo com a nova coluna 'emergente'
    df_macro.to_csv(output_path, index=True)
    
    print(f"Resultados salvos com sucesso em: {output_path}")
    
    return df_macro

def intersecao_emergentes_tanimoto_cosseno(df_tanimoto, df_cosseno):
    """Retorna DataFrame com todas as patentes (união) e coluna 'emergente'
    = True apenas se emergente em TANIMOTO E em COSSENO."""
    if df_tanimoto is None and df_cosseno is None:
        print("Ambos os DataFrames são None. Retornando vazio.")
        return pd.DataFrame()

    df_t = df_tanimoto.copy() if df_tanimoto is not None else pd.DataFrame(columns=['lens_id'])
    df_c = df_cosseno.copy() if df_cosseno is not None else pd.DataFrame(columns=['lens_id'])

    df_t['lens_id'] = df_t['lens_id'].astype(str)
    df_c['lens_id'] = df_c['lens_id'].astype(str)

    for df in (df_t, df_c):
        if 'emergente' not in df.columns:
            df['emergente'] = False
        if 'patentes_similares' not in df.columns:
            df['patentes_similares'] = [[] for _ in range(len(df))]

    df_t = df_t.rename(columns={
        'emergente': 'emergente_tanimoto',
        'patentes_similares': 'patentes_similares_tanimoto'
    })
    df_c = df_c.rename(columns={
        'emergente': 'emergente_cosseno',
        'patentes_similares': 'patentes_similares_cosseno'
    })

    df_union = pd.merge(df_t, df_c, on='lens_id', how='outer', suffixes=('_t', '_c'))

    import ast
    def _to_list_safe(x):
        if isinstance(x, list):
            return x
        if pd.isna(x):
            return []
        try:
            val = ast.literal_eval(x) if isinstance(x, str) else x
            if isinstance(val, list):
                return val
            return [val]
        except Exception:
            return [str(x)]

    def _merge_unique_lists(a, b):
        out = []
        for item in (a or []) + (b or []):
            s = str(item)
            if s not in out:
                out.append(s)
        return out
    df_union['emergente_tanimoto'] = df_union.get('emergente_tanimoto', False).fillna(False).astype(bool)
    df_union['emergente_cosseno'] = df_union.get('emergente_cosseno', False).fillna(False).astype(bool)

    df_union['patentes_similares_tanimoto'] = df_union.get('patentes_similares_tanimoto', []).apply(_to_list_safe)
    df_union['patentes_similares_cosseno'] = df_union.get('patentes_similares_cosseno', []).apply(_to_list_safe)

    df_union['emergente'] = df_union['emergente_tanimoto'] & df_union['emergente_cosseno']

    df_union['patentes_similares'] = df_union.apply(
        lambda r: _merge_unique_lists(r['patentes_similares_tanimoto'], r['patentes_similares_cosseno']),
        axis=1
    )

    cols_keep = ['lens_id', 'emergente', 'patentes_similares',
                 'emergente_tanimoto', 'emergente_cosseno',
                 'patentes_similares_tanimoto', 'patentes_similares_cosseno']
    existing = [c for c in cols_keep if c in df_union.columns]
    result = df_union[existing].copy()

    return result


#### Funções para Carregar resultados

In [ ]:
def parse_set_safe(val):
    """Converte string representando set para tipo set Python."""
    if pd.isna(val):
        return set()
    try:
        result = ast.literal_eval(val)
        return set(result) if isinstance(result, set) else set(result)
    except (ValueError, SyntaxError, TypeError):
        return set()

def parse_list_safe(val):
    """Converte string representando lista para tipo list Python."""
    if pd.isna(val):
        return []
    try:
        result = ast.literal_eval(val)
        return result if isinstance(result, list) else list(result)
    except (ValueError, SyntaxError, TypeError):
        return []

def parse_dict_safe(val):
    """Converte string representando dicionário para tipo dict Python."""
    if pd.isna(val):
        return {}
    try:
        result = ast.literal_eval(val)
        return result if isinstance(result, dict) else {}
    except (ValueError, SyntaxError, TypeError):
        return {}

def carregar_analise_jaccard(numero_palavras_por_topico, percentual_similaridade, 
                             anos_a_frente=1):
    """Carrega resultados da análise de similaridade Jaccard."""
    caminho = (
        f'{ANALYSIS_DIR}analise-de-similaridade_jaccard_'
        f'n{numero_palavras_por_topico}_s{percentual_similaridade:.2f}_'
        f'anos{anos_a_frente}.csv'
    )
    
    converters = {
        'palavras_chave': parse_set_safe,
        'patentes_similares': parse_list_safe
    }
    
    dtypes = {
        'lens_id': 'string',
        'year': 'Int64',
        'count_similares': 'Int64',
        'emergente': 'boolean'
    }
    
    print(f"Carregando arquivo: {caminho}...")
    
    try:
        df = pd.read_csv(
            caminho,
            index_col=0,
            converters=converters,
            dtype=dtypes,
            low_memory=False
        )
        
        print("\nArquivo carregado com sucesso!")
        print(f"Total de registros: {len(df):,}")
        
        return df
        
    except FileNotFoundError:
        print(f"ERRO: Arquivo não encontrado")
        return None
    except Exception as e:
        print(f"ERRO: {e}")
        return None

def carregar_analise_tanimoto_ou_cosseno(numero_palavras_por_topico, percentual_similaridade, 
                                         metodo='tanimoto', anos_a_frente=1):
    """Carrega resultados da análise de similaridade Tanimoto/Cossenos."""
    caminho = (
        f'{ANALYSIS_DIR}analise-de-similaridade_{metodo}_'
        f'n{numero_palavras_por_topico}_s{percentual_similaridade:.2f}_'
        f'anos{anos_a_frente}.csv'
    )
    
    converters = {
        'vetor_caracteristicas': parse_dict_safe,
        'patentes_similares': parse_list_safe
    }
    
    dtypes = {
        'lens_id': 'string',
        'year': 'Int64',
        'count_similares': 'Int64',
        'emergente': 'boolean'
    }
    
    print(f"Carregando arquivo: {caminho}...")
    
    try:
        df = pd.read_csv(
            caminho,
            index_col=0,
            converters=converters,
            dtype=dtypes,
            low_memory=False
        )
        
        print("\nArquivo carregado com sucesso!")
        print(f"Total de registros: {len(df):,}")
        
        return df
        
    except FileNotFoundError:
        print(f"ERRO: Arquivo não encontrado")
        return None
    except Exception as e:
        print(f"ERRO: {e}")
        return None

def carregar_analise_topicos(numero_palavras_por_topico, percentual_similaridade,
                            metodo='tanimoto', anos_a_frente=1):
    """
    Carrega resultados da análise de patentes vs tópicos (analise_emergencia_patente_topico).

    Retorna DataFrame ou None em caso de erro.
    """
    caminho = (
        f'{ANALYSIS_DIR}/analise-de-similaridade_topicos_{metodo}_'
        f'n{numero_palavras_por_topico}_s{percentual_similaridade:.2f}_'
        f'anos{anos_a_frente}.csv'
    )

    print(f"Carregando arquivo: {caminho}...")

    try:
        df = pd.read_csv(caminho, index_col=0, low_memory=False)
        # Converte colunas Topic_* que foram salvas como JSON/string de volta para listas
        try:
            df = converter_json_para_listas(df, prefixo_coluna='Topic_')
        except Exception:
            # fallback: aplica parse_list_safe nas colunas que comecem com Topic_
            topic_cols = [c for c in df.columns if c.startswith('Topic_')]
            for col in topic_cols:
                df[col] = df[col].apply(parse_list_safe)

        # Garante que patentes_similares seja lista
        if 'patentes_similares' in df.columns:
            df['patentes_similares'] = df['patentes_similares'].apply(parse_list_safe)

        # Normaliza dtypes comuns
        if 'emergente' in df.columns:
            try:
                df['emergente'] = df['emergente'].astype('boolean')
            except Exception:
                df['emergente'] = df['emergente'].map(lambda x: bool(x) if pd.notna(x) else pd.NA).astype('boolean')

        if 'count_similares' in df.columns:
            df['count_similares'] = pd.to_numeric(df['count_similares'], errors='coerce').astype('Int64')

        print("\nArquivo carregado com sucesso!")
        print(f"Total de registros: {len(df):,}")

        return df

    except FileNotFoundError:
        print("ERRO: Arquivo não encontrado")
        return None
    except Exception as e:
        print(f"ERRO: {e}")
        return None



#### Funções deAnalise com Classificação do IPCR

In [ ]:
def safe_convert_list_to_strings(input_data):
    """Converte dados para lista de strings de forma segura."""
    lista_bruta = []
    
    try:
        if isinstance(input_data, str):
            lista_bruta = ast.literal_eval(input_data)
        elif isinstance(input_data, list):
            lista_bruta = input_data
        
        if isinstance(lista_bruta, list):
            return [str(item).strip() for item in lista_bruta if pd.notna(item)]
        else:
            return []
            
    except (ValueError, SyntaxError, TypeError):
        return []


def analisar_com_ipcr(df_main, filtered_ipcr):
    """Analisa patentes emergentes verificando classificação IPCR."""
    df_lookup = filtered_ipcr[['lens_id', 'new_ipcr']].copy()
    df_lookup['lens_id'] = df_lookup['lens_id'].astype(str).str.strip()
    
    patentes_com_new_ipcr = set(
        df_lookup[df_lookup['new_ipcr'] == True]['lens_id']
    )
    
    df_resultado = df_main.copy()
    df_resultado['emergent_new_ipcr'] = False
    
    idx_emergente = df_resultado[df_resultado['emergente']].index
    
    s_similares_lista = df_resultado.loc[idx_emergente, 'patentes_similares'].apply(
        safe_convert_list_to_strings
    )
    
    s_explodida = s_similares_lista.explode().dropna()
    s_matches = s_explodida.isin(patentes_com_new_ipcr)
    
    df_comparacoes = pd.DataFrame({
        'patente_similar_verificada': s_explodida,
        'teve_new_ipcr': s_matches
    })
    df_comparacoes['patente_original_lens_id'] = df_comparacoes.index.map(
        df_resultado['lens_id']
    )
    df_comparacoes = df_comparacoes[[
        'patente_original_lens_id', 
        'patente_similar_verificada', 
        'teve_new_ipcr'
    ]].reset_index(drop=True)
    
    s_resultado_final = s_matches.groupby(level=0).any()
    df_resultado.loc[s_resultado_final.index, 'emergent_new_ipcr'] = s_resultado_final
    
    total_emergent = df_resultado['emergent_new_ipcr'].sum()
    print(f"\nTotal de 'emergent_new_ipcr' = True: {total_emergent}")
    
    return df_resultado, df_comparacoes



#### Funções de treinamento Machine Learning (SVM e XGBOOST)

In [ ]:
def treinar_modelos_ml(df_data, df_labels, config,
                       text_features=['title_abstract', 'inventor_names'],
                       numeric_features=['year', 'patent_count', 'family_count', 'claims_count'],
                       categorical_features=['kind', 'publication_type', 'patent_status'],
                       use_new_ipcr_class=True,
                       save_models=False,
                       save_native=False,
                       save_results=False,
                       save_models_dir=MODEL_ML_DIR,
                       save_results_dir='./resultados_analise/ml/'):
    """
    Treina e avalia modelos SVM e XGBoost com pré-processamento e
    múltiplas estratégias de balanceamento/otimização.
    
    Args:
        df_data: DataFrame com dados das patentes.
        df_labels: DataFrame com labels (emergent_new_ipcr) or (emergente).
        config: Dicionário com flags de configuração:
            'SMOTE_apply' (bool): Usa SMOTE para oversampling.
            'USE_GRID_SEARCH_SVM' (bool): Usa GridSearchCV para otimizar o SVM.
            'USE_GRID_SEARCH_XGB' (bool): Usa GridSearchCV para otimizar o XGBoost.
            'USE_GRID_SEARCH_PARAMS' (bool): Usa parâmetros otimizados pré-definidos.
    """
    
    # --- 0. Constante de Aleatoriedade ---
    # Usando 42 para consistência com o teu primeiro script
    RANDOM_STATE = 42 
    
    print(f"--- Dados Iniciais (Patentes): {df_data.shape} ---")
    print(f"--- Rótulos Iniciais (Labels): {df_labels.shape} ---")
    
    # --- 1. Preparação dos Dados ---
    df_merged = pd.merge(df_data, df_labels, on='lens_id')
    print(f"\n--- Dados Combinados: {df_merged.shape} ---")
    
    df_merged[text_features] = df_merged[text_features].fillna('')
    df_merged[numeric_features] = df_merged[numeric_features].fillna(0)
    df_merged[categorical_features] = df_merged[categorical_features].fillna('Missing').infer_objects(copy=False)
    
    X = df_merged[text_features + numeric_features + categorical_features]
    if use_new_ipcr_class:
        y_raw = df_merged['emergent_new_ipcr']
    else:
        y_raw = df_merged['emergente']

    le = LabelEncoder()
    y = le.fit_transform(y_raw)
    
    # --- 2. Engenharia de Features (Pré-processamento) ---
    text_transformer = TfidfVectorizer(
        max_features=5000, 
        ngram_range=(1, 2)
    )
    numeric_transformer = StandardScaler()
    categorical_transformer = OneHotEncoder(handle_unknown='ignore')
    
    preprocessor = ColumnTransformer(
        transformers=[
            ('text', text_transformer, text_features[0]),
            ('num', numeric_transformer, numeric_features),
            ('cat', categorical_transformer, categorical_features)
        ]
    )
    
    # --- 3. Divisão dos Dados (Treino e Teste) ---
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
    )
    
    count_neg = np.sum(y_train == 0)
    count_pos = np.sum(y_train == 1)
    # Evitar divisão por zero se não houver positivos (embora raro)
    scale_weight = count_neg / count_pos if count_pos > 0 else 1 
    
    print(f"\nDivisão dos dados de TREINO:")
    print(f"  Classe Negativa (False): {count_neg}")
    print(f"  Classe Positiva (True):  {count_pos}")
    print(f"  Proporção (scale_pos_weight): {scale_weight:.2f}")

    # Variáveis para guardar resultados e modelos
    svm_pipeline = None
    y_pred_svm = None
    svm_best_params = None
    
    xgb_pipeline = None
    y_pred_xgb = None
    xgb_best_params = None

    # --- 4. Modelo 1: Treinamento SVM ---
    
    if config.get('SMOTE_apply', False):
        print("\n--- Treinando SVM (com SMOTE) ---")
        svm_pipeline = ImbPipeline(steps=[
            ('preprocessor', preprocessor),
            ('smote', SMOTE(random_state=RANDOM_STATE)),
            ('classifier', SVC(kernel='linear', random_state=RANDOM_STATE))
        ])
        svm_pipeline.fit(X_train, y_train)
        y_pred_svm = svm_pipeline.predict(X_test)

    elif config.get('USE_GRID_SEARCH_SVM', False):
        print("\n--- Treinando SVM (Otimizado com GridSearchCV) ---")
        svm_pipeline_for_grid = Pipeline(steps=[
            ('preprocessor', preprocessor),
            ('classifier', SVC(random_state=RANDOM_STATE, class_weight='balanced'))
        ])
        
        svm_param_grid = {
            'classifier__kernel': ['linear', 'rbf'],
            'classifier__C': [0.1, 1, 10]
        }
        
        svm_grid_search = GridSearchCV(estimator=svm_pipeline_for_grid,
                                       param_grid=svm_param_grid,
                                       cv=3,
                                       scoring='f1_macro',
                                       n_jobs=-1,
                                       verbose=2)
        
        print("Iniciando o GridSearch para o SVM...")
        svm_grid_search.fit(X_train, y_train)
        
        svm_best_params = svm_grid_search.best_params_
        svm_pipeline = svm_grid_search.best_estimator_ 
        
        print("Avaliando o melhor modelo SVM...")
        y_pred_svm = svm_pipeline.predict(X_test)
        
    elif config.get('USE_GRID_SEARCH_PARAMS', False):
        print("\n--- Treinando SVM (Balanceado com Parâmetros Otimizados) ---")
        svm_pipeline = Pipeline(steps=[
            ('preprocessor', preprocessor),
            ('classifier', SVC(
                kernel='rbf',
                random_state=RANDOM_STATE,
                class_weight='balanced',
                C=1
            ))
        ])
        svm_pipeline.fit(X_train, y_train)
        y_pred_svm = svm_pipeline.predict(X_test)

    else:
        print("\n--- Treinando SVM (Balanceado - Padrão) ---")
        svm_pipeline = Pipeline(steps=[
            ('preprocessor', preprocessor),
            ('classifier', SVC(
                kernel='linear',
                random_state=RANDOM_STATE,
                class_weight='balanced'
            ))
        ])
        svm_pipeline.fit(X_train, y_train)
        y_pred_svm = svm_pipeline.predict(X_test)

    
    # --- 5. Modelo 2: Treinamento XGBoost ---
    
    if config.get('SMOTE_apply', False):
        print("\n--- Treinando XGBoost (com SMOTE) ---")
        xgb_pipeline = ImbPipeline(steps=[
            ('preprocessor', preprocessor),
            ('smote', SMOTE(random_state=RANDOM_STATE)),
            ('classifier', XGBClassifier(
                use_label_encoder=False,
                eval_metric='logloss',
                random_state=RANDOM_STATE,
                n_estimators=100
            ))
        ])
        xgb_pipeline.fit(X_train, y_train)
        y_pred_xgb = xgb_pipeline.predict(X_test)

    elif config.get('USE_GRID_SEARCH_XGB', False):
        print("\n--- Treinando XGBoost (Otimizado com GridSearchCV) ---")
        xgb_pipeline_for_grid = Pipeline(steps=[
            ('preprocessor', preprocessor),
            ('classifier', XGBClassifier(
                use_label_encoder=False,
                eval_metric='logloss',
                random_state=RANDOM_STATE,
                scale_pos_weight=scale_weight
            ))
        ])
        
        param_grid = {
            'classifier__n_estimators': [100, 250, 500],
            'classifier__max_depth': [3, 5, 7],
            'classifier__learning_rate': [0.1, 0.05]
        }
        
        grid_search = GridSearchCV(estimator=xgb_pipeline_for_grid,
                                   param_grid=param_grid,
                                   cv=3,
                                   scoring='f1_macro',
                                   n_jobs=-1,
                                   verbose=2)
        
        print("Iniciando o GridSearch para o XGBoost...")
        grid_search.fit(X_train, y_train)
        
        xgb_best_params = grid_search.best_params_
        xgb_pipeline = grid_search.best_estimator_ # Salva o melhor modelo
        
        print("Avaliando o melhor modelo XGBoost...")
        y_pred_xgb = xgb_pipeline.predict(X_test)
        
    elif config.get('USE_GRID_SEARCH_PARAMS', False):
        print("\n--- Treinando XGBoost (Balanceado com Parâmetros Otimizados) ---")
        xgb_pipeline = Pipeline(steps=[
            ('preprocessor', preprocessor),
            ('classifier', XGBClassifier(
                use_label_encoder=False,
                eval_metric='logloss',
                random_state=RANDOM_STATE,
                n_estimators=500,
                scale_pos_weight=scale_weight,
                max_depth=3,
                learning_rate=0.1
            ))
        ])
        xgb_pipeline.fit(X_train, y_train)
        y_pred_xgb = xgb_pipeline.predict(X_test)

    else:
        print("\n--- Treinando XGBoost (Balanceado - Padrão) ---")
        xgb_pipeline = Pipeline(steps=[
            ('preprocessor', preprocessor),
            ('classifier', XGBClassifier(
                use_label_encoder=False,
                eval_metric='logloss',
                random_state=RANDOM_STATE,
                n_estimators=100,
                scale_pos_weight=scale_weight
            ))
        ])
        xgb_pipeline.fit(X_train, y_train)
        y_pred_xgb = xgb_pipeline.predict(X_test)
    
    # --- 6. Avaliação dos Modelos ---
    target_names = le.classes_.astype(str)
    report_lines = []
    
    report_lines.append("="*80)
    report_lines.append("RESULTADOS FINAIS DA AVALIAÇÃO")
    report_lines.append("="*80)
    
    # --- Relatório SVM ---
    report_lines.append("\n--- Modelo SVM ---")
    if y_pred_svm is not None:
        if config.get('USE_GRID_SEARCH_SVM', False) and svm_best_params:
            report_lines.append(f"Melhores Parâmetros Encontrados: {svm_best_params}")
        
        svm_accuracy = accuracy_score(y_test, y_pred_svm)
        svm_report_str = classification_report(y_test, y_pred_svm, target_names=target_names)
        
        report_lines.append(f"Acurácia: {svm_accuracy:.4f}")
        report_lines.append("Relatório de Classificação:")
        report_lines.append(svm_report_str)
    else:
        report_lines.append("Modelo SVM não foi treinado.")

    # --- Relatório XGBoost ---
    report_lines.append("\n--- Modelo XGBoost ---")
    if y_pred_xgb is not None:
        if config.get('USE_GRID_SEARCH_XGB', False) and xgb_best_params:
            report_lines.append(f"Melhores Parâmetros Encontrados: {xgb_best_params}")
            
        xgb_accuracy = accuracy_score(y_test, y_pred_xgb)
        xgb_report_str = classification_report(y_test, y_pred_xgb, target_names=target_names)
            
        report_lines.append(f"Acurácia: {xgb_accuracy:.4f}")
        report_lines.append("Relatório de Classificação:")
        report_lines.append(xgb_report_str)
    else:
        report_lines.append("Modelo XGBoost não foi treinado.")
    
    report_lines.append("\n" + "="*80)
    
    final_report_string = "\n".join(report_lines)
    
    print("\n\n" + final_report_string)
    
    # --- 7. Salvamento de Resultados e Modelos ---
    try:
        df_test_full = df_merged.loc[X_test.index].reset_index(drop=True)
    except Exception:
        df_test_full = pd.DataFrame(X_test).reset_index(drop=True)
    try:
        y_true_labels = le.inverse_transform(y_test)
    except Exception:
        y_true_labels = y_test

    resultados_df = df_test_full.copy()
    resultados_df['y_true'] = pd.Series(y_true_labels).reset_index(drop=True)

    if y_pred_svm is not None:
        try:
            y_pred_svm_labels = le.inverse_transform(y_pred_svm)
        except Exception:
            y_pred_svm_labels = y_pred_svm
        resultados_df['y_pred_svm'] = pd.Series(y_pred_svm_labels).reset_index(drop=True)
    else:
        resultados_df['y_pred_svm'] = pd.NA

    if y_pred_xgb is not None:
        try:
            y_pred_xgb_labels = le.inverse_transform(y_pred_xgb)
        except Exception:
            y_pred_xgb_labels = y_pred_xgb
        resultados_df['y_pred_xgb'] = pd.Series(y_pred_xgb_labels).reset_index(drop=True)
    else:
        resultados_df['y_pred_xgb'] = pd.NA
    
    if save_results:
        try:
            os.makedirs(save_results_dir, exist_ok=True)
            csv_fname = f'result_ml_predicoes.csv'
            resultados_path = os.path.join(save_results_dir, csv_fname)
            resultados_df.to_csv(resultados_path, index=False)
            print(f'✓ Resultados (predições) salvos em: {resultados_path}')
            report_fname = f'report_ml_avaliacao.txt'
            report_path = os.path.join(save_results_dir, report_fname)
            with open(report_path, 'w', encoding='utf-8') as f:
                f.write(final_report_string)
            print(f'✓ Relatório de Avaliação (texto) salvo em: {report_path}')
            
        except Exception as e:
            print(f'Erro ao salvar resultados: {e}')
    
    if save_models:
        try:
            os.makedirs(save_models_dir, exist_ok=True)
            timestamp = int(time.time())
            if svm_pipeline is not None:
                svm_path = os.path.join(save_models_dir, f'svm_pipeline_{timestamp}.joblib')
                joblib.dump(svm_pipeline, svm_path)
                print(f'✓ SVM pipeline salvo em: {svm_path}')
                if save_native:
                    try:
                        if hasattr(svm_pipeline, 'named_steps') and 'classifier' in svm_pipeline.named_steps:
                            svm_clf = svm_pipeline.named_steps['classifier']
                        elif hasattr(svm_pipeline, 'steps'):
                            svm_clf = svm_pipeline.steps[-1][1]
                        else:
                            svm_clf = svm_pipeline
                        svm_clf_path = os.path.join(save_models_dir, f'svm_classifier_{"newipcr" if use_new_ipcr_class else "std"}_{timestamp}.pkl')
                        with open(svm_clf_path, 'wb') as f:
                            pickle.dump(svm_clf, f)
                        print(f'✓ SVM classifier (pickle) salvo em: {svm_clf_path}')
                    except Exception as e:
                        print(f'Erro ao salvar SVM nativo: {e}')
            if xgb_pipeline is not None:
                xgb_path = os.path.join(save_models_dir, f'xgb_pipeline_{"newipcr" if use_new_ipcr_class else "std"}_{timestamp}.joblib')
                joblib.dump(xgb_pipeline, xgb_path)
                print(f'✓ XGBoost pipeline salvo em: {xgb_path}')
                if save_native:
                    try:
                        if hasattr(xgb_pipeline, 'named_steps') and 'classifier' in xgb_pipeline.named_steps:
                            xgb_clf = xgb_pipeline.named_steps['classifier']
                        elif hasattr(xgb_pipeline, 'steps'):
                            xgb_clf = xgb_pipeline.steps[-1][1]
                        else:
                            xgb_clf = xgb_pipeline
                        if hasattr(xgb_clf, 'get_booster'):
                            xgb_native_path = os.path.join(save_models_dir, f'xgb_booster_{"newipcr" if use_new_ipcr_class else "std"}_{timestamp}.json')
                            xgb_clf.get_booster().save_model(xgb_native_path)
                            print(f'✓ XGBoost booster (nativo) salvo em: {xgb_native_path}')
                        else:
                            print('Aviso: classificador XGBoost não possui get_booster() — não foi possível salvar nativamente.')
                    except Exception as e:
                        print(f'Erro ao salvar XGBoost nativo: {e}')
        except Exception as e:
            print(f'Erro ao salvar modelos: {e}')

    return svm_pipeline, xgb_pipeline, (X_test, y_test, y_pred_svm, y_pred_xgb)

## Execução do código

### Configurações de Execução

In [ ]:
# CONFIGURAÇÕES
# NOTA: é possivel que algumas conbinações de flags resultem em erros ou
# comportamentos inesperados.

SAMPLE_REDUCED_DATASET = True # Usar dataset reduzido para testes rápidos

# LDA
TREINAR_MODELOS = False # Treino LDA

PROCESSAR_PATENTES = True # se já tiver sido processada para aquela quantidade de topicos não refaz

# ANALISES DE SIMILARIDADE
EXECUTAR_ANALISE_JACCARD = False
EXECUTAR_ANALISE_TANIMOTO = False
EXECUTAR_ANALISE_COSSENO = False
EXECUTAR_ANALISE_TOPICOS = False
ANOS_A_FRENTE = 3 # anos futuros para comparar
PERCENTUAL_SIMILARIDADE = 0.80
NUM_PALAVRAS_POR_TOPICO = 5

# ML
USE_IPCR_FOR_ML = True # Usar classificação IPCR para rotular patentes emergentes

USE_JACCARD_FOR_ML = False
USE_TANIMOTO_FOR_ML = False
USE_COSSENO_FOR_ML = False
USE_TOPICOS_FOR_ML = False # não é executado com a analise de IPCR

# Usar interseção das análises (tanimoto e cosseno) para rotular patentes emergentes
USE_INTERSECAO_FOR_ML = True
 
EXECUTAR_ML = True # Treino ML
SAVE_RESULTADOS_ML = True
SAVE_MODELS_ML = False

# Todos False = configuração padrão (balanceado simples)
config_ml = {
        'SMOTE_apply': False,             # Usar SMOTE para balanceamento (oversampling)
        'USE_GRID_SEARCH_SVM': True,     # Usar GridSearchCV para SVM
        'USE_GRID_SEARCH_XGB': True,     # Usar GridSearchCV para XGBoost
        'USE_GRID_SEARCH_PARAMS': False    # Usar os melhores parâmetros fixos
}

# Features iniciaispara ML
# TEXT_FEATURES = ['title_abstract', 'inventor_names']
# NUMERIC_FEATURES = ['year', 'patent_count', 'family_count', 'claims_count']
# CATEGORICAL_FEATURES = ['kind', 'publication_type', 'patent_status']

TEXT_FEATURES = ['title_abstract', 'inventor_names'] 

NUMERIC_FEATURES = [
    'year', 
    'patent_count', 
    'family_count', 
    'claims_count',
    'num_cpc_codes',          
    'num_applicants',       
    'num_priority_claims'   
]

CATEGORICAL_FEATURES = [
    'kind', 
    'publication_type', 
    'patent_status',
    'new_ipcr',
    'picked',                
    'has_foreign_priority'
]

### Carregar Dados

In [ ]:
print("="*80)
print("CARREGANDO DADOS")
print("="*80)

df_novo = load_dataset(DATASET_PATH)
print(f"Dataset carregado: {len(df_novo)} patentes")

# Adiciona coluna 'year' se não existir
if 'year' not in df_novo.columns:
    df_novo['year'] = df_novo['date_published'].dt.year

# Remove coluna 'index' se existir
if 'index' in df_novo.columns:
    df_novo = df_novo.drop(columns=['index'])

if SAMPLE_REDUCED_DATASET:
    df_novo = sample_reduce_by_year(df_novo, keep_fraction=0.15, seed=42)
    print(df_novo['year'].value_counts().sort_index())
    print(f"Dataset reduzido: {len(df_novo)} patentes")

df_novo = df_novo.reset_index(drop=True)
print(f"Colunas: {df_novo.columns.tolist()}")

### Treinamento LDA

In [ ]:
if TREINAR_MODELOS:
    print("\n" + "="*80)
    print("TREINAMENTO DE MODELOS LDA")
    print("="*80)
    
    train_and_save_lda_per_year(
        df=df_novo,
        num_topics=NUMERO_DE_TOPICOS,
        passes=10,
        output_dir=MODELS_DIR
    )
    
    # Exibe os tópicos dos modelos treinados
    print("\n--- Visualizando Tópicos dos Modelos ---")
    display_topics_from_models_with_probs(MODELS_DIR, num_words=WORDS_PER_TOPIC_FOR_DISPLAY)
else:
    print("\n" + "="*80)
    print("TREINAMENTO PULADO (modelos já existem)")
    display_topics_from_models_with_probs(MODELS_DIR, num_words=WORDS_PER_TOPIC_FOR_DISPLAY)
    print("="*80)

### Processamento das Patentes

In [ ]:
if PROCESSAR_PATENTES:
    print("\n" + "="*80)
    print("PROCESSAMENTO DE PATENTES")
    print("="*80)
    
    analisar_e_salvar_por_ano_aprimorado(
        df=df_novo,
        modelos_dir=MODELS_DIR,
        num_topics=NUMERO_DE_TOPICOS,
        output_dir=RESULTS_DIR,
        ignorar_existentes=True
    )
else:
    print("\n" + "="*80)
    print("PROCESSAMENTO PULADO (resultados já existem)")
    print("="*80)

### Dataframe com os Resultados (df_completo)

In [ ]:
print("\n" + "="*80)
print("CONCATENANDO RESULTADOS")
print("="*80)

df_completo = concatenar_resultados_por_ano(
    diretorio_resultados=RESULTS_DIR,
    formato='auto'
)

if not df_completo.empty:
    # Converte colunas JSON para listas
    df_completo = converter_json_para_listas(df_completo)
    print("\nDados concatenados e convertidos com sucesso!")
    print(f"Total de patentes: {len(df_completo)}")
else:
    print("\nErro: Nenhum dado foi concatenado!")
    raise ValueError("Falha ao concatenar resultados")

topic_cols = [c for c in df_completo.columns if c.startswith('Topic_')]

if topic_cols:
    # chama a função definida em outro local passando topic_cols corretamente
    df_completo['dominant_topic'] = df_completo.apply(
        lambda row: _compute_dominant_topic(row, topic_cols), axis=1
    )
    print("\nColuna 'dominant_topic' adicionada. Distribuição:")
    print(df_completo['dominant_topic'].value_counts(dropna=False).head(10))
else:
    df_completo['dominant_topic'] = None
    print("\nNenhuma coluna 'Topic_' encontrada — 'dominant_topic' criada com None.")


### Analise de Patentes Jaccard

In [ ]:
if EXECUTAR_ANALISE_JACCARD:
    print("\n" + "="*80)
    print("ANÁLISE DE SIMILARIDADE (JACCARD)")
    print("="*80)
    
    df_analise_jaccard = analise_similaridade_jaccard(
        df_completo=df_completo,
        numero_palavras_por_topico=NUM_PALAVRAS_POR_TOPICO,
        percentual_similaridade=PERCENTUAL_SIMILARIDADE,
        anos_a_frente=ANOS_A_FRENTE
    )
    print("Análise Jaccard concluída")
else:
    print("\n" + "="*80)
    print("CARREGANDO ANÁLISE JACCARD EXISTENTE")
    print("="*80)
    
    df_analise_jaccard = carregar_analise_jaccard(
        numero_palavras_por_topico=NUM_PALAVRAS_POR_TOPICO,
        percentual_similaridade=PERCENTUAL_SIMILARIDADE,
        anos_a_frente=ANOS_A_FRENTE
    )
    if df_analise_jaccard is None:
        print("Erro ao carregar análise Jaccard!")
    else:
        print("Análise Jaccard carregada")

### Analise de Patentes Tanimoto

In [ ]:
if EXECUTAR_ANALISE_TANIMOTO:
        print("\n" + "="*80)
        print(f"ANÁLISE DE SIMILARIDADE (TANIMOTO)")
        print("="*80)
        
        df_analise_tanimoto = analise_similaridade_tanimoto_or_cosseno(
            df_completo=df_completo,
            numero_palavras_por_topico=NUM_PALAVRAS_POR_TOPICO,
            percentual_similaridade=PERCENTUAL_SIMILARIDADE,
            metodo='tanimoto',
            anos_a_frente=ANOS_A_FRENTE
        )
        print(f"Análise tanimoto concluída")
else:
        print("\n" + "="*80)
        print(f"CARREGANDO ANÁLISE TANIMOTO EXISTENTE")
        print("="*80)
        
        df_analise_tanimoto = carregar_analise_tanimoto_ou_cosseno(
            numero_palavras_por_topico=NUM_PALAVRAS_POR_TOPICO,
            percentual_similaridade=PERCENTUAL_SIMILARIDADE,
            metodo='tanimoto',
            anos_a_frente=ANOS_A_FRENTE
        )
        if df_analise_tanimoto is None:
            print(f"Erro ao carregar análise tanimoto!")
        else:
            print(f"Análise tanimoto carregada")

### Análise de Patentes Cossenos

In [ ]:
if EXECUTAR_ANALISE_COSSENO:
    print("\n" + "="*80)
    print(f"ANÁLISE DE SIMILARIDADE (COSSENOS)")
    print("="*80)
    
    df_analise_cosseno = analise_similaridade_tanimoto_or_cosseno(
        df_completo=df_completo,
        numero_palavras_por_topico=NUM_PALAVRAS_POR_TOPICO,
        percentual_similaridade=PERCENTUAL_SIMILARIDADE,
        metodo='cossenos',
        anos_a_frente=ANOS_A_FRENTE
    )
    print(f"Análise cossenos concluída")
else:
    print("\n" + "="*80)
    print(f"CARREGANDO ANÁLISE COSSENO EXISTENTE")
    print("="*80)
    
    df_analise_cosseno = carregar_analise_tanimoto_ou_cosseno(
        numero_palavras_por_topico=NUM_PALAVRAS_POR_TOPICO,
        percentual_similaridade=PERCENTUAL_SIMILARIDADE,
        metodo='cossenos',
        anos_a_frente=ANOS_A_FRENTE
    )
    if df_analise_cosseno is None:
        print(f"Erro ao carregar análise cossenos!")
    else:
        print(f"Análise cossenos carregada")

### Análise de Patentes por Tópico

In [ ]:
if EXECUTAR_ANALISE_TOPICOS:
    print("\n" + "="*80)
    print(f"ANÁLISE DE PATENTES VS TÓPICOS FUTUROS")
    print("="*80)
    
    df_analise_topicos = analise_emergencia_patente_topico(
        df_completo=df_completo,
        numero_palavras_por_topico=NUM_PALAVRAS_POR_TOPICO,
        metodo='cossenos',
        anos_a_frente=ANOS_A_FRENTE,
        percentual_similaridade=PERCENTUAL_SIMILARIDADE
    )
else:
    print("\n" + "="*80)
    print(f"ANÁLISE DE PATENTES VS TÓPICOS FUTUROS PULADA")
    print("="*80)
    
    df_analise_topicos = carregar_analise_topicos(
        numero_palavras_por_topico=NUM_PALAVRAS_POR_TOPICO,
        metodo='cossenos',
        anos_a_frente=ANOS_A_FRENTE,
        percentual_similaridade=PERCENTUAL_SIMILARIDADE
    )
    if df_analise_topicos is None:
        print("Erro ao carregar análise de tópicos!")
    else:
        print("Análise de tópicos carregada com sucesso!")

### Calcular a intersecção entre tanimoto e cosseno

In [ ]:
if USE_INTERSECAO_FOR_ML:
    print("\n" + "="*80)
    print("CALCULANDO INTERSEÇÃO DAS ANÁLISES PARA ML")
    print("="*80)
    df_intersection = intersecao_emergentes_tanimoto_cosseno(df_analise_tanimoto, df_analise_cosseno)
    print("Numero de patentes na interseção:", int(df_intersection['emergente'].sum()))

### Análise junto da classificação do IPCR

In [ ]:
if USE_IPCR_FOR_ML:
    print("\n" + "="*80)
    print("ANÁLISE COM CLASSIFICAÇÃO IPCR")
    print("="*80)

    # Carrega dados IPCR
    filtered_ipcr = pd.read_csv(FILTERED_IPCR_PATH, compression='zip')
    print(f"Dataset IPCR carregado: {len(filtered_ipcr)} registros")

    if df_analise_jaccard is not None:
        # Análise com IPCR - Jaccard
        print("\n--- Analisando patentes emergentes (Jaccard) ---")
        df_resultado_jaccard, df_comparacoes_jaccard = analisar_com_ipcr(
            df_main=df_analise_jaccard,
            filtered_ipcr=filtered_ipcr
        )
        print("Análise IPCR (Jaccard) concluída")

    if df_analise_tanimoto is not None:
        # Análise com IPCR - Tanimoto
        print("\n--- Analisando patentes emergentes (Tanimoto) ---")
        df_resultado_tanimoto, df_comparacoes_tanimoto = analisar_com_ipcr(
            df_main=df_analise_tanimoto,
            filtered_ipcr=filtered_ipcr
        )
        print("Análise IPCR (Tanimoto) concluída")

    if df_analise_cosseno is not None:
        # Análise com IPCR - Cossenos
        print("\n--- Analisando patentes emergentes (Cossenos) ---")
        df_resultado_cosseno, df_comparacoes_cosseno = analisar_com_ipcr(
            df_main=df_analise_cosseno,
            filtered_ipcr=filtered_ipcr
        )
        print("Análise IPCR (Cossenos) concluída")

    if USE_INTERSECAO_FOR_ML and 'df_intersection' in locals():
        if df_intersection is not None:
            # Análise com IPCR - Interseção
            print("\n--- Analisando patentes emergentes (Interseção) ---")
            df_resultado_intersection, df_comparacoes_intersection = analisar_com_ipcr(
                df_main=df_intersection,
                filtered_ipcr=filtered_ipcr
            )
            print("Análise IPCR (Interseção) concluída")
else:
    print("\n" + "="*80)
    print("ANÁLISE COM CLASSIFICAÇÃO IPCR PULADA")
    print("="*80)

    print("Utilizando DataFrames originais para ML (sem IPCR)")

### Preparação para o ML

In [ ]:
print("\n" + "="*80)
print("PREPARANDO DADOS PARA ML")
print("="*80)

# Merge com dados completos para ML
colunas_para_ml = ['lens_id', 'family_count', 'claims_count', 
                'ipcr_publication_years', 'new_ipcr']

if USE_IPCR_FOR_ML:
    colunas_existentes = [col for col in colunas_para_ml if col in filtered_ipcr.columns]
    
    df_train = pd.merge(
        df_novo,
        filtered_ipcr[colunas_existentes],
        on='lens_id',
        how='left'
    )
else:
    df_train = df_novo.copy()
print(f"Dataset de treino preparado: {df_train.shape}")

### Processar dados para o ML

In [ ]:
df_train['biblio_obj'] = df_train['biblio'].apply(parse_dict_safe)

df_train['cpc_obj'] = df_train['cpc_triples'].apply(parse_list_safe)

print("A extrair novas features...")

# Feature 1: num_cpc_codes
df_train['num_cpc_codes'] = df_train['cpc_obj'].apply(len)

# Feature 2: num_applicants
df_train['num_applicants'] = df_train['biblio_obj'].apply(
    lambda x: len(x.get('parties', {}).get('applicants', []))
)

# Feature 3: num_priority_claims
df_train['num_priority_claims'] = df_train['biblio_obj'].apply(
    lambda x: len(x.get('priority_claims', {}).get('claims', []))
)

# Feature 4: has_foreign_priority (Booleano: True/False)
def check_foreign_priority(biblio_dict):
    claims_list = biblio_dict.get('priority_claims', {}).get('claims', [])
    if not claims_list:
        return False
    
    for claim in claims_list:
        if claim.get('jurisdiction') != 'BR':
            return True
    return False

df_train['has_foreign_priority'] = df_train['biblio_obj'].apply(check_foreign_priority)

print("Novas features criadas com sucesso!")
print(df_train[['num_cpc_codes', 'num_applicants', 'num_priority_claims', 'has_foreign_priority']].head())

# remover o ano de 1999 e 2024 do dataset de treino (estes anos possuem muitos campos com 'missing')
linhas_antes = len(df_train)
print(f"Número de linhas original: {linhas_antes}")

df_train = df_train[df_train['year'] != 1999]
df_train = df_train[df_train['year'] != 2024]
linhas_depois = len(df_train)
print(f"Número de linhas após remoção de 1999 e 2024: {linhas_depois} (removidas {linhas_antes - linhas_depois})")

print("\n" + "="*80)
print("Dataset de treino final preparado")
print("="*80)

### Treinamento ML (SVM e XGBoost)

In [ ]:
if EXECUTAR_ML:
    print("\n" + "="*80)
    print("TREINAMENTO DE MODELOS ML")
    print("="*80)
    
    if USE_IPCR_FOR_ML:
        if USE_JACCARD_FOR_ML:
            print("\n--- Treinando modelos ML (Jaccard com IPCR) ---")
            df_labels = df_resultado_jaccard[['lens_id', 'emergent_new_ipcr']].copy()

            svm_model_jaccard, xgb_model_jaccard, resultados_jaccard = treinar_modelos_ml(
                df_data=df_train,
                df_labels=df_labels,
                config=config_ml,
                use_new_ipcr_class=True,
                text_features=TEXT_FEATURES,
                numeric_features=NUMERIC_FEATURES,
                categorical_features=CATEGORICAL_FEATURES,
                save_models=SAVE_MODELS_ML,
                save_native=SAVE_MODELS_ML,
                save_results=SAVE_RESULTADOS_ML
            )
        if USE_TANIMOTO_FOR_ML:
            print("\n--- Treinando modelos ML (Tanimoto com IPCR) ---")
            df_labels = df_resultado_tanimoto[['lens_id', 'emergent_new_ipcr']].copy()
            
            svm_model_tanimoto, xgb_model_tanimoto, resultados_tanimoto = treinar_modelos_ml(
                df_data=df_train,
                df_labels=df_labels,
                config=config_ml,
                use_new_ipcr_class=True,
                text_features=TEXT_FEATURES,
                numeric_features=NUMERIC_FEATURES,
                categorical_features=CATEGORICAL_FEATURES,
                save_models=SAVE_MODELS_ML,
                save_native=SAVE_MODELS_ML,
                save_results=SAVE_RESULTADOS_ML
            )
        if USE_COSSENO_FOR_ML:
            print("\n--- Treinando modelos ML (Cosseno com IPCR) ---")
            df_labels = df_resultado_cosseno[['lens_id', 'emergent_new_ipcr']].copy()
            
            svm_model_cosseno, xgb_model_cosseno, resultados_cosseno = treinar_modelos_ml(
                df_data=df_train,
                df_labels=df_labels,
                config=config_ml,
                use_new_ipcr_class=True,
                text_features=TEXT_FEATURES,
                numeric_features=NUMERIC_FEATURES,
                categorical_features=CATEGORICAL_FEATURES,
                save_models=SAVE_MODELS_ML,
                save_native=SAVE_MODELS_ML,
                save_results=SAVE_RESULTADOS_ML
            )
        if USE_INTERSECAO_FOR_ML:
            print("\n--- Treinando modelos ML (Interseção com IPCR) ---")
            df_labels = df_resultado_intersection[['lens_id', 'emergent_new_ipcr']].copy()
            
            svm_model_intersection, xgb_model_intersection, resultados_intersection = treinar_modelos_ml(
                df_data=df_train,
                df_labels=df_labels,
                config=config_ml,
                use_new_ipcr_class=True,
                text_features=TEXT_FEATURES,
                numeric_features=NUMERIC_FEATURES,
                categorical_features=CATEGORICAL_FEATURES,
                save_models=SAVE_MODELS_ML,
                save_native=SAVE_MODELS_ML,
                save_results=SAVE_RESULTADOS_ML
            )
        print("\nTreinamento de modelos ML concluído!")
    else:
        if USE_JACCARD_FOR_ML:
            print("\n--- Treinando modelos ML (Jaccard sem IPCR) ---")
            df_labels = df_analise_jaccard[['lens_id', 'emergente']].copy()

            svm_model_jaccard, xgb_model_jaccard, resultados_jaccard = treinar_modelos_ml(
                df_data=df_train,
                df_labels=df_labels,
                config=config_ml,
                use_new_ipcr_class=False,
                text_features=TEXT_FEATURES,
                numeric_features=NUMERIC_FEATURES,
                categorical_features=CATEGORICAL_FEATURES,
                save_models=SAVE_MODELS_ML,
                save_native=SAVE_MODELS_ML,
                save_results=SAVE_RESULTADOS_ML
            )
        if USE_TANIMOTO_FOR_ML:
            print("\n--- Treinando modelos ML (Tanimoto sem IPCR) ---")
            df_labels = df_analise_tanimoto[['lens_id', 'emergente']].copy()
            
            svm_model_tanimoto, xgb_model_tanimoto, resultados_tanimoto = treinar_modelos_ml(
                df_data=df_train,
                df_labels=df_labels,
                config=config_ml,
                use_new_ipcr_class=False,
                text_features=TEXT_FEATURES,
                numeric_features=NUMERIC_FEATURES,
                categorical_features=CATEGORICAL_FEATURES,
                save_models=SAVE_MODELS_ML,
                save_native=SAVE_MODELS_ML,
                save_results=SAVE_RESULTADOS_ML
            )
        if USE_COSSENO_FOR_ML:
            print("\n--- Treinando modelos ML (Cosseno sem IPCR) ---")
            df_labels = df_analise_cosseno[['lens_id', 'emergente']].copy()
            
            svm_model_cosseno, xgb_model_cosseno, resultados_cosseno = treinar_modelos_ml(
                df_data=df_train,
                df_labels=df_labels,
                config=config_ml,
                use_new_ipcr_class=False,
                text_features=TEXT_FEATURES,
                numeric_features=NUMERIC_FEATURES,
                categorical_features=CATEGORICAL_FEATURES,
                save_models=SAVE_MODELS_ML,
                save_native=SAVE_MODELS_ML,
                save_results=SAVE_RESULTADOS_ML
            )
        if USE_TOPICOS_FOR_ML:
            df_labels = df_analise_topicos[['lens_id', 'emergente']].copy()
            
            # se labels não tiver nenhum true ou for toda "False", não treina
            if df_labels['emergente'].nunique() <= 1:
                print("\nAviso: A coluna 'emergente' contém apenas uma classe. O treinamento será pulado para análise de tópicos.")
            else:
                svm_model_topicos, xgb_model_topicos, resultados_topicos = treinar_modelos_ml(
                    df_data=df_train,
                    df_labels=df_labels,
                    config=config_ml,
                    use_new_ipcr_class=False,
                    text_features=TEXT_FEATURES,
                    numeric_features=NUMERIC_FEATURES,
                    categorical_features=CATEGORICAL_FEATURES,
                    save_models=SAVE_MODELS_ML,
                    save_native=SAVE_MODELS_ML,
                save_results=SAVE_RESULTADOS_ML
                )
        if USE_INTERSECAO_FOR_ML:
            print("\n--- Treinando modelos ML (Interseção sem IPCR) ---")
            df_labels = df_intersection[['lens_id', 'emergente']].copy()
            
            svm_model_intersection, xgb_model_intersection, resultados_intersection = treinar_modelos_ml(
                df_data=df_train,
                df_labels=df_labels,
                config=config_ml,
                use_new_ipcr_class=True,
                text_features=TEXT_FEATURES,
                numeric_features=NUMERIC_FEATURES,
                categorical_features=CATEGORICAL_FEATURES,
                save_models=SAVE_MODELS_ML,
                save_native=SAVE_MODELS_ML,
                save_results=SAVE_RESULTADOS_ML
            )
        print("\nTreinamento de modelos ML concluído!")
else:
    print("\n" + "="*80)
    print("TREINAMENTO ML PULADO")
    print("="*80)

### Resumo

In [ ]:
print("="*80)
print("RESUMO DA EXECUÇÃO")
print("="*80)
print(f" - Dados carregados: {len(df_novo):,} patentes")
print(f" - Modelos LDA: {NUMERO_DE_TOPICOS} tópicos")
print(f" - Análise de similaridade: {NUM_PALAVRAS_POR_TOPICO} palavras, {PERCENTUAL_SIMILARIDADE:.0%} threshold")

if not df_completo.empty:
    print(f" - Resultados concatenados: {len(df_completo):,} patentes")
if 'df_analise_jaccard' in locals() and df_analise_jaccard is not None:
    emergentes_j = df_analise_jaccard['emergente'].sum()
    print(f" - Patentes emergentes (Jaccard): {emergentes_j:,}")
if 'df_analise_tanimoto' in locals() and df_analise_tanimoto is not None:
    emergentes_t = df_analise_tanimoto['emergente'].sum()
    print(f" - Patentes emergentes (tanimoto): {emergentes_t:,}")
if 'df_analise_cosseno' in locals() and df_analise_cosseno is not None:
    emergentes_c = df_analise_cosseno['emergente'].sum()
    print(f" - Patentes emergentes (cosseno): {emergentes_c:,}")
if 'df_analise_topicos' in locals() and df_analise_topicos is not None:
    emergentes_topicos = df_analise_topicos['emergente'].sum()
    print(f" - Patentes emergentes (tópicos): {emergentes_topicos:,}")
if 'df_intersection' in locals() and df_intersection is not None:
    emergentes_i = df_intersection['lens_id'].nunique()
    print(f" - Patentes emergentes (interseção): {emergentes_i:,}")
if 'df_resultado_jaccard' in locals() and df_resultado_jaccard is not None:
    ipcr_j = df_resultado_jaccard['emergent_new_ipcr'].sum()
    print(f" - Emergentes com novo IPCR (Jaccard): {ipcr_j:,}")
if 'df_resultado_tanimoto' in locals() and df_resultado_tanimoto is not None:
    ipcr_t = df_resultado_tanimoto['emergent_new_ipcr'].sum()
    print(f" - Emergentes com novo IPCR (Tanimoto): {ipcr_t:,}")
if 'df_resultado_cosseno' in locals() and df_resultado_cosseno is not None:
    ipcr_c = df_resultado_cosseno['emergent_new_ipcr'].sum()
    print(f" - Emergentes com novo IPCR (Cosseno): {ipcr_c:,}")
if 'df_resultado_intersection' in locals() and df_resultado_intersection is not None:
    ipcr_i = df_resultado_intersection['emergent_new_ipcr'].sum()
    print(f" - Emergentes com novo IPCR (Interseção): {ipcr_i:,}")
print("="*80)
print("EXECUÇÃO CONCLUÍDA COM SUCESSO!")
print("="*80)